# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/asadh-74/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

**Lane:** Lane 2 — Refresh / Content Opportunity Scoring (carried over from `w01_research_question.ipynb`).

**Skills loaded for this task:** `skills/writing-data-contracts/SKILL.md` + `skills/flyrank/flyrank-data/SKILL.md` (per `skills/README.md`).

Work the sections in order. Simple words, honest numbers, three queries, five features, one deliberate leak.

## 0. Connect to the warehouse

DuckDB reads the release straight off Hugging Face — nothing is downloaded in full. Token comes from a Colab Secret named `HF_TOKEN` (never pasted in a cell — this repo is public).

In [ ]:
%pip -q install duckdb

import os, getpass
import duckdb
import pandas as pd

# Token order: env var -> Colab Secret -> prompt (last resort, avoid on Colab reconnects)
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
MONTH = '2026-03'  # mid-panel month, per the assignment warning — never the _sample table for label logic

FACT_MONTH = f"read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"
DIM_CLIENTS = f"read_parquet('{REL}/dim_clients.parquet')"

# Sanity check the columns actually on this build before trusting any query below
con.sql(f'DESCRIBE SELECT * FROM {FACT_MONTH} LIMIT 1').show()


## 1. Unit of analysis + time window

**One row = one content page (`content_hash_id`, within one `client_hash_id`), summarized over one calendar month.** The daily fact table's own grain is report_date × client × content — I aggregate that up to content × month, because Lane 2's decision ("review this page or not") is made once a month, not once a day.

**Table(s):** `fact_content_daily_performance` (the `month=2026-03` partition only, for iteration — the `_sample` table is excluded on purpose since it's June 2026, the panel's last month) joined to `dim_content` for static page attributes (`content_type`, `word_count`, `content_created_date`). `dim_clients` is used for context only (per-client `gsc_data_start`/`ga4_data_start`), never as a feature source.

**Time window:** 2026-03-01 through 2026-03-31 — a mid-panel month, chosen deliberately instead of the sealed final month (June 2026) so I never develop label logic inside my own future test window.

In [ ]:
# Verified below in Section 3, query 2 (row count + date span)


## 2. Fields: feature / label / context / excluded

**What I'd predict or rank (label / proxy):** `review_candidate` — a page that is visible in search (meaningful March impressions) but converting almost none of that visibility into clicks. This is a *proxy* for "a human should look at this page," not ground truth that the page actually needs a refresh — I can't observe FlyRank's own product decision (it isn't shipped in this data), only the observable signals a refresh decision would be based on.

| Field | Bucket | Why |
|---|---|---|
| `gsc_impressions` (summed over March) | Feature | Fully inside the March window, closed before the April 1 decision moment |
| `gsc_clicks` (summed over March) | Feature | Same — observed, closed, in-window |
| `gsc_avg_position` (averaged over March) | Feature | Same — observed, closed, in-window |
| `ctr_month` (clicks/impressions, derived) | Feature | Derived only from this month's own closed numbers, no future peeking |
| `content_created_date` → `content_age_days` | Feature | Static registration fact, set long before March, from `dim_content` |
| `content_hash_id`, `client_hash_id` | Context | Scrambled IDs — join/group/split only, never model input |
| `ga4_data_available` | Context (gate) | Used to filter which rows have trustworthy GA4 columns, not fed to a model |
| `review_candidate` (the label) | Label / proxy | Computed FROM `ctr_month` + `gsc_impressions` — never a feature |
| `sessions_ai` (AI-referral sessions) | **Excluded (deliberately)** | Warehouse-wide only 30,177 of 78,835,655 rows carry AI-session activity — too sparse to support a Lane-2 review signal this month; belongs to the separate AI Referral Opportunity lane |


In [ ]:
# Field classification is a design decision, not a query — verified implicitly by which
# columns appear (or deliberately don't appear) in the feature frame built in Section 3.


## 3. Verify it with queries (grain, counts, availability) — then five features — then the trap

### Query 1 — grain
Prove the base fact table really is one row per `report_date × client_hash_id × content_hash_id`, on the March partition. Zero rows back means the grain holds.

In [ ]:
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {FACT_MONTH}
    GROUP BY 1, 2, 3
    HAVING c > 1
    LIMIT 5
""").df()

print(f'duplicate (date, client, content) rows found: {len(grain_check)}')
grain_check


### Query 2 — slice row count + date span
How many rows are in my March slice, how many distinct pages does it cover, and does the date span match what March 2026 should look like?

In [ ]:
slice_summary = con.sql(f"""
    SELECT
        COUNT(*)                       AS row_count,
        COUNT(DISTINCT content_hash_id) AS distinct_pages,
        COUNT(DISTINCT client_hash_id)  AS distinct_clients,
        MIN(report_date)                AS min_date,
        MAX(report_date)                AS max_date
    FROM {FACT_MONTH}
""").df()

slice_summary


### Query 3 — availability, filtered with `IS TRUE`
`ga4_data_available` is three-valued (`TRUE` / `FALSE` / `NULL`) per the FlyRank data dictionary — rows can be genuinely GA4-tracked, explicitly not-yet-tracked, or simply unresolved. A plain `= TRUE` or `NOT ga4_data_available` silently mishandles the NULLs, so I filter with `IS TRUE` / `IS NOT TRUE` and show how many rows survive each way.

In [ ]:
availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END)     AS ga4_available_true,
        SUM(CASE WHEN ga4_data_available IS NOT TRUE THEN 1 ELSE 0 END) AS ga4_not_true,
        SUM(CASE WHEN ga4_data_available IS NULL THEN 1 ELSE 0 END)    AS ga4_null
    FROM {FACT_MONTH}
""").df()

availability


### Five features (max) — the March feature frame for Lane 2

Built entirely from March 2026, aggregated per page. Every feature line states *why* it was already knowable at the decision moment (April 1, 2026, looking back at closed March data).

In [ ]:
features = con.sql(f"""
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        SUM(f.gsc_impressions)                         AS imp_month,
        SUM(f.gsc_clicks)                               AS clicks_month,
        AVG(f.gsc_avg_position)                          AS avg_position_month,
        SUM(f.gsc_clicks) * 1.0 / NULLIF(SUM(f.gsc_impressions), 0) AS ctr_month,
        ANY_VALUE(c.content_created_date)                  AS content_created_date
    FROM {FACT_MONTH} f
    LEFT JOIN {DIM_CONTENT} c ON f.content_hash_id = c.content_hash_id
    GROUP BY 1, 2
    HAVING imp_month >= 10
""").df()

features['content_age_days'] = (pd.Timestamp('2026-04-01') - pd.to_datetime(features['content_created_date'])).dt.days

print(f'{len(features):,} pages in the March feature frame')
features.head()


**Five features, one line each — "knowable at the decision moment because…":**

1. **`imp_month`** — search impressions summed over March 2026. Knowable because March is fully closed by April 1; nothing here reaches into April.
2. **`clicks_month`** — clicks summed over March 2026. Same reasoning — closed, observed, in-window.
3. **`avg_position_month`** — average SERP position across March. Also a closed-window observed signal, not a forward-looking one.
4. **`ctr_month`** — `clicks_month / imp_month`, derived purely from the two features above. A derived measurement of an already-knowable window is still knowable — no new information sneaks in.
5. **`content_age_days`** — days between `content_created_date` (a static, one-time registration fact from `dim_content`) and the April 1 decision date. The page's birthdate is fixed long before March even starts.


### The trap — add one label-derived column on purpose

First, define the proxy label from this month's own closed numbers: a page counts as a `review_candidate` if it's visible (≥100 impressions) but converting almost none of that visibility (`ctr_month` below a small threshold). Then I train a quick, honest baseline on the five real features above. Then I add ONE cheat column that is just the label's own rule wearing a feature costume — watch the score jump toward 1.0 — then delete it and keep the honest number.

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

model_data = features.dropna(subset=['ctr_month', 'avg_position_month', 'content_age_days']).copy()
model_data['review_candidate'] = (
    (model_data['imp_month'] >= 100) & (model_data['ctr_month'] < 0.005)
).astype(int)

print('review_candidate rate:', model_data['review_candidate'].mean().round(3))

honest_features = ['imp_month', 'clicks_month', 'avg_position_month', 'ctr_month', 'content_age_days']
X = model_data[honest_features]
y = model_data['review_candidate']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
honest_model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
honest_auc = roc_auc_score(y_te, honest_model.predict_proba(X_te)[:, 1])
print(f'HONEST score (5 real features only), ROC-AUC: {honest_auc:.3f}')


In [ ]:
# --- THE TRAP: add one column derived straight from the label's own rule ---
model_data_leaky = model_data.copy()
model_data_leaky['leak_low_ctr_flag'] = (model_data_leaky['ctr_month'] < 0.005).astype(int)
# ^ this is literally half of how review_candidate was defined above — a label-derived column,
#   not a genuinely new observation.

leaky_features = honest_features + ['leak_low_ctr_flag']
Xl = model_data_leaky[leaky_features]
yl = model_data_leaky['review_candidate']

Xl_tr, Xl_te, yl_tr, yl_te = train_test_split(Xl, yl, test_size=0.25, random_state=42, stratify=yl)
leaky_model = LogisticRegression(max_iter=1000).fit(Xl_tr, yl_tr)
leaky_auc = roc_auc_score(yl_te, leaky_model.predict_proba(Xl_te)[:, 1])
print(f'LEAKY score (with leak_low_ctr_flag), ROC-AUC: {leaky_auc:.3f}')
print(f'Jump: {honest_auc:.3f} -> {leaky_auc:.3f}')


In [ ]:
# --- Delete the leak, keep the honest number ---
del model_data_leaky['leak_low_ctr_flag']

print(f'FINAL honest score kept for this contract: ROC-AUC = {honest_auc:.3f}')
print('The leaky column is removed. It never goes into any downstream feature frame.')


**The lesson (notebook 02, reproduced here on real warehouse data):** `leak_low_ctr_flag` didn't add any observation the model didn't already have — it just handed back a thresholded copy of the label itself. The AUC jump is fake confidence, not real predictive power. The number that ships in this contract is the honest one, from the five features that were actually knowable before the decision moment.

## 4. Data limits

**Named limitation:** this slice can't tell a genuine drop in page-level demand apart from an under-tracked client-month. Panel depth is unbalanced across clients (`dim_clients.gsc_data_start` / `ga4_data_start` differ per client), and `ga4_data_available` can be `NULL` rather than a clean `TRUE`/`FALSE` (Query 3 above shows the split). A page that looks like a weak `review_candidate` in March might just belong to a client whose GA4 tracking hadn't started yet — that's a tracking gap, not a content problem, and this month-level slice alone can't tell the two apart without checking `dim_clients` first.

In [ ]:
# Named limitation is a design/read of Query 3's output above — no new query needed here.


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all) — **run this in Colab with your `HF_TOKEN` secret before committing; outputs above are not yet executed**
- [x] No client names, URLs, or private queries anywhere — only hashed IDs and aggregate numbers
- [x] My claims use careful words: proxy, observed, closed window, knowable-at-decision-moment
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.